# Semana 1 · Sesión 1: Jupyter y notebooks reproducibles

**Módulo 0** · Facultad de Ciencias, UNAM

## Objetivos de la sesión

1. Moverse con soltura en JupyterLab: celdas, magics y extensiones.
2. Diagnosticar problemas de orden de ejecución en un notebook.
3. Exportar un notebook a otros formatos con `nbconvert`.

## Antes de empezar

Esta clase asume que ya instalaste tu entorno siguiendo
[`docs/instalacion.md`](../../docs/instalacion.md) y el checklist de
[`preparacion.md`](../preparacion/preparacion.md). Si algo de eso no
funcionó, dilo ahora — lo resolvemos antes de seguir.

Hoy trabajamos con **Python puro y Jupyter**: no vamos a importar ninguna
librería. NumPy y Matplotlib entran en la sesión 2 de esta misma semana.

## Celdas de código vs. markdown

Un notebook combina dos tipos de celda:

- **Markdown** (como esta): texto, ecuaciones en LaTeX, explicación del
  razonamiento.
- **Código**: lo que se ejecuta.

La regla del curso: cada celda de código va acompañada de markdown que
explica el *por qué*, no solo el *qué*. El resultado de un cálculo sin
contexto no sirve como material reproducible — ni para ti en tres meses, ni
para quien revise tu PR.

## Magics útiles

Los *magics* son comandos especiales de Jupyter/IPython, no Python puro
(empiezan con `%` o `%%`):

| Magic | Para qué sirve |
|---|---|
| `%timeit` | Mide el tiempo de una línea, promediando varias corridas |
| `%%timeit` | Igual, pero para toda la celda |
| `%matplotlib inline` | Muestra las figuras de Matplotlib dentro del notebook |
| `%whos` | Lista las variables definidas hasta el momento, con su tipo |

No son parte del lenguaje Python — no funcionan si exportas la celda a un
script `.py` sin quitarlos primero (ver la sección de exportación, más
abajo).

In [ ]:
# Dos formas de construir la misma lista de cuadrados: ¿cuál es más rápida?
%timeit cuadrados = [i**2 for i in range(10_000)]

## TODO en clase 1

Compara, con `%%timeit` (una celda por enfoque, ya que el magic mide toda la
celda), dos formas de sumar los primeros $10^6$ enteros:

1. Con la función incorporada `sum(range(10**6))`.
2. Con un `for` que acumula el resultado en una variable.

¿Cuál es más rápida? ¿Por qué crees que pasa eso?

In [ ]:
# TODO en clase: mide con %%timeit el enfoque con sum()

In [ ]:
# TODO en clase: mide con %%timeit el enfoque con un for acumulador

## `%whos`: qué hay vivo en el kernel

Un notebook no ejecuta en el vacío: cada celda que corres deja variables
en la memoria del *kernel*, y ahí se quedan aunque borres la celda que las
creó. `%whos` lista todo lo que está vivo en este momento, con su tipo.

Es la herramienta más directa para responder "¿por qué esta variable vale
lo que vale?" — y, como veremos en un momento, para detectar que estás
usando algo que ya no deberías tener.

In [ ]:
masa = 2.5              # kg
velocidad = 3.0         # m/s
etiquetas = ["antes", "después"]

%whos

## TODO en clase 2

Ejecuta lo siguiente y observa cómo cambia la salida de `%whos`:

1. Borra la variable `masa` con `del masa`.
2. Vuelve a llamar `%whos`.

¿Desapareció de la lista? Ahora vuelve a ejecutar **la celda de arriba**,
la que define `masa`. ¿Qué pasa con el número de ejecución `[n]` a la
izquierda de cada celda?

In [ ]:
# TODO en clase: borra masa con del y vuelve a listar las variables vivas

## Extensiones útiles de JupyterLab

No las vamos a instalar en vivo (la instalación vive en
`docs/instalacion.md`, y en clase no usamos `!pip install`). Solo para que
las conozcas:

- **Table of Contents** — navega el notebook por sus encabezados `##`.
- **Variable Inspector** — muestra las variables activas sin escribir
  `%whos` cada vez.
- **Spellchecker** — revisa ortografía en las celdas de markdown.

## Buenas prácticas: reproducibilidad y orden de ejecución

Los notebooks permiten ejecutar celdas en cualquier orden — y eso es
también su mayor riesgo. Un notebook que "funciona" en tu sesión actual
puede fallar para cualquier otra persona si el orden real de ejecución no
coincide con el orden en que aparecen las celdas.

| Síntoma | Causa típica |
|---|---|
| `NameError` al reabrir el notebook | Una variable se definió en una celda que luego borraste o moviste |
| El resultado cambia según quién lo corre | Una celda de arriba se ejecutó dos veces (p. ej. un contador que se incrementa) |
| Funciona en tu máquina, falla en la revisión | Nunca se probó desde un kernel limpio |

Por eso, antes de dar un notebook por terminado (y **siempre** antes de
abrir un PR): `Kernel → Restart & Run All`. Si eso no corre limpio de
principio a fin, el notebook no está listo.

## El error más común, en vivo

La tabla de arriba describe el síntoma; vamos a provocarlo. El caso
clásico: defines una variable en una celda, la usas más abajo, y después
**borras o mueves** esa primera celda. Mientras el kernel siga vivo, todo
parece funcionar — la variable sigue en memoria. El notebook solo se rompe
para la siguiente persona que lo abra desde cero.

Aquí simulamos ese "desde cero" con `del`, que quita la variable del
kernel igual que lo haría un reinicio:

In [ ]:
resultado_parcial = 42
del resultado_parcial   # equivale a: esa celda ya no existe, o nunca se ejecutó

try:
    print(resultado_parcial * 2)
except NameError as error:
    print("NameError:", error)

## TODO en clase 3

Ese `NameError` es exactamente lo que verá quien revise tu notebook si el
orden de las celdas no corresponde al orden en que las ejecutaste.

La cura no es un truco de código, es un hábito: **`Kernel → Restart & Run
All`** antes de dar cualquier notebook por terminado.

Hazlo ahora mismo con este notebook y confirma que llega hasta el final
sin errores. Después, en la celda de abajo, escribe en un comentario en
qué número de ejecución `[n]` quedó la primera celda de código.

In [ ]:
# TODO en clase: tras Restart & Run All, anota aquí el [n] de la primera celda

## Exportación: `nbconvert`

A veces conviene convertir un notebook a otro formato: un script `.py` para
correrlo fuera de Jupyter, o HTML para compartirlo con alguien que no tiene
Jupyter instalado. La herramienta es `nbconvert`, desde la terminal (no
desde una celda del notebook — en este curso no usamos `!` para comandos de
shell dentro de un notebook):

```bash
jupyter nbconvert --to script mi_notebook.ipynb   # -> mi_notebook.py
jupyter nbconvert --to html mi_notebook.ipynb     # -> mi_notebook.html
```

Es también lo que usamos para *verificar* que un notebook ejecuta limpio
antes de un PR:

```bash
jupyter nbconvert --to notebook --execute --inplace mi_notebook.ipynb
```

## TODO en clase 4

Los magics (`%timeit`, `%whos`) **no son Python**: si exportas el notebook
a un script, no sobreviven como los escribiste. Compruébalo.

1. Desde la terminal, exporta este notebook a un script:
   `jupyter nbconvert --to script <nombre>.ipynb`
2. Abre el `.py` que se generó y busca las líneas donde estaban tus
   magics. ¿En qué se convirtieron?
3. En la celda de abajo, lee ese archivo desde Python y cuenta cuántas
   líneas **empiezan** con `get_ipython` (usa `str.startswith`). Ojo:
   si buscas la palabra en cualquier posición te va a contar también
   los comentarios de este enunciado, que quedaron dentro del `.py`.

Esa es la razón por la que un notebook lleno de magics no se puede correr
tal cual fuera de Jupyter.

In [ ]:
# TODO en clase: lee el .py exportado y cuenta las líneas que empiezan con get_ipython
from pathlib import Path

ruta_script = ...
cuantos_magics = ...

## Resumen

Hoy trabajamos el entorno: celdas de código y markdown, magics
(`%timeit`, `%whos`), extensiones de JupyterLab, y por qué el orden de
ejecución es la fuente de errores más común en un notebook. También vimos
cómo exportar con `nbconvert` y por qué los magics no sobreviven al
export.

La regla que se queda con nosotros todo el curso: **`Restart & Run All`
antes de dar un notebook por terminado.**

**Próxima sesión — Semana 1, sesión 2:** NumPy y Matplotlib, el repaso
numérico que sirve de contraste con el cómputo simbólico.